In [1]:
from utils import load_data, split_train_val
from torchvision.transforms import v2 
import torch 

from models import CNNClassifier, ResNet, ResNet50Transfer, save_model
from torch.optim import Adam, lr_scheduler

from torch.nn import CrossEntropyLoss 
import random 

import numpy as np 

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Mean: tensor([0.6940, 0.6538, 0.6292])
# Std: tensor([0.2249, 0.2396, 0.2480])

transform = v2.Compose([
    v2.ToImage(), # ToTensor() PIL IMAGE -> Tensor (H, W, C)
    v2.ToDtype(torch.float32, scale=True), # (C, H, W) [0-255] -> # (C, H, W) [0-1]
    v2.Resize((124, 142)), # necesitamos que todas las imagenes tengan la misma dimension
    # Data Augmentation 
    v2.RandomHorizontalFlip(p=0.1),
    v2.RandomVerticalFlip(p=0.1),
    v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    v2.RandomRotation(degrees=10),
    v2.Normalize(mean=[0.6940, 0.6538, 0.6292], std=[0.2249, 0.2396, 0.2480]),
])

split_train_val()

train_loader = load_data("./train_dataset.csv", transform=transform, batch_size=100)

transform_test = v2.Compose([
    v2.ToImage(), # ToTensor() PIL IMAGE -> Tensor (H, W, C)
    v2.ToDtype(torch.float32, scale=True), # (C, H, W) [0-255] -> # (C, H, W) [0-1]
    v2.Resize((124, 142)), # necesitamos que todas las imagenes tengan la misma dimension
    v2.Normalize(mean=[0.6940, 0.6538, 0.6292], std=[0.2249, 0.2396, 0.2480]),
])

test_loader = load_data("./test_dataset.csv", transform=transform_test, batch_size=100)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [2]:
#model = CNNClassifier()
#model = ResNet()
model = ResNet50Transfer()
model.to(device)

learning_rate = 1e-4
weight_decay = 1e-4
criterion = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/ramiro/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 101MB/s] 


In [3]:
best_acc = float('-inf')

patience_count = 0
max_patience = 10

num_epochs = 100

for epoch in range(num_epochs):
    running_loss = 0
    correct_train = 0
    total_train = 0

    for (X, y) in train_loader:
        X = X.to(device) # [N, 3, 140, 162]
        y = y.to(device) # [N]

        outputs = model(X) # [N, 10]

        # Input [N, 10] Target [N]
        loss = criterion(outputs, y)
        running_loss += loss.item()

        # Calcular accuracy
        _, predicted = torch.max(outputs.data, dim=1)
        total_train += y.size(0)
        correct_train += (predicted==y).sum().item()

        # Actualizar los pesos
        optimizer.zero_grad()
        loss.backward() # recalcula los gradientes
        optimizer.step() # recalcula los parametros

    epoch_train_acc = 100 * correct_train / total_train
    epoch_loss = running_loss / len(train_loader)

    # TODO: Verificar el rendimiento actual del modelo sobre el test_set
    # Calcular Test Accuracy

    correct_test = 0
    total_test = 0 
    with torch.no_grad():
        for (X, y) in test_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            _, predicted = torch.max(outputs.data, dim=1)
            total_test += y.size(0)
            correct_test += (predicted==y).sum().item()

    epoch_test_acc = 100 * correct_test / total_test

    # Early Stopping
    if epoch_test_acc > best_acc:
        best_acc = epoch_test_acc
        patience_count = 0  
        save_model(model, f'{model._get_name()}_{best_acc:.0f}.th')
    else:
        patience_count += 1
    
    scheduler.step(epoch_test_acc)

    print(f"Epoch {epoch+1}/{num_epochs}. Loss: {epoch_loss:.4f} - Train Accuracy: {epoch_train_acc:.2f}% - Test Accuracy: {epoch_test_acc:.2f}%")

    if patience_count == max_patience:
        break

Epoch 1/100. Loss: 2.2687 - Train Accuracy: 13.99% - Test Accuracy: 30.94%
Epoch 2/100. Loss: 2.1702 - Train Accuracy: 38.50% - Test Accuracy: 57.18%
Epoch 3/100. Loss: 2.0487 - Train Accuracy: 58.59% - Test Accuracy: 66.57%
Epoch 4/100. Loss: 1.9132 - Train Accuracy: 69.25% - Test Accuracy: 77.07%
Epoch 5/100. Loss: 1.7514 - Train Accuracy: 76.18% - Test Accuracy: 79.28%
Epoch 6/100. Loss: 1.5909 - Train Accuracy: 79.29% - Test Accuracy: 80.66%
Epoch 7/100. Loss: 1.4454 - Train Accuracy: 82.06% - Test Accuracy: 81.77%
Epoch 8/100. Loss: 1.3112 - Train Accuracy: 82.76% - Test Accuracy: 80.11%
Epoch 9/100. Loss: 1.2083 - Train Accuracy: 83.31% - Test Accuracy: 81.77%
Epoch 10/100. Loss: 1.0946 - Train Accuracy: 84.83% - Test Accuracy: 84.81%
Epoch 11/100. Loss: 1.0302 - Train Accuracy: 84.63% - Test Accuracy: 83.43%
Epoch 12/100. Loss: 0.9577 - Train Accuracy: 85.18% - Test Accuracy: 82.04%
Epoch 13/100. Loss: 0.8772 - Train Accuracy: 86.91% - Test Accuracy: 83.98%
Epoch 14/100. Loss: 0